In [2]:
from dotenv import load_dotenv
load_dotenv('../.env')

from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2")

response = llm.invoke("llama3.2")

response = llm.invoke("What is 2847 multiplied by 193")
print("Without tools:")
print(response.content)
print()
print("─" * 60)
print()

# LLMs are notoriously bad at arithmetic
# They pattern-match from training data rather than actually calculating
# The answer should be 549,471
print("Correct answer: 2847 × 193 =", 2847 * 193)
print()
print("This is why agents need tools.")
print("The LLM decides WHAT to do. The tool actually does it.")

Without tools:
The result of multiplying 2847 by 193 is:

2847 × 193 = 550,011

────────────────────────────────────────────────────────────

Correct answer: 2847 × 193 = 549471

This is why agents need tools.
The LLM decides WHAT to do. The tool actually does it.


In [5]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
import math

llm = ChatOllama(model="llama3.2")

# ── @tool decorator turns any Python function into a LangChain tool ────────
# The docstring is critical — it tells the LLM what the tool does
# and when to use it. Write it for the LLM, not for humans.

@tool
def calculate(expression : str) -> str:
    """
    Evaluate a mathematical expression and return the result.
    Use this tool whenever the user asks for any calculation, arithmetic,
    percentage, or numerical computation.
    Input must be a valid Python math expression like '2847 * 193' or '15 / 100 * 5000'.
    """

    try:
        result = eval(expression, {"__builtins__": {}}, {
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow, "sqrt": math.sqrt,
            "pi": math.pi, "e": math.e
        })
        return f"{expression} = {result}"
    except Exception as ex:
        return f"Error evaluating expression: {ex}"
    
@tool
def get_word_count(text: str) -> str:
    """
    Count the number of words in a piece of text.
    Use this when the user asks how many words something contains.
    """
    count = len(text.split())
    return f"Word count: {count} words"

@tool
def reverse_text(text: str) -> str:
    """
    Reverse a piece of text.
    Use this when the user asks to reverse or flip text.
    """
    return text[::-1]

# ── Inspect what a tool looks like to the LLM ─────────────────────────────
print("Tools defined. Here's what the LLM sees:\n")
for t in [calculate, get_word_count, reverse_text]:
    print(f"Tool: {t.name}")
    print(f"Description: {t.description}")
    print(f"Input schema: {t.args}")
    print()


llm_with_tools = llm.bind_tools([calculate, get_word_count, reverse_text])

test_messages = [
    "What is 2847 multiplied by 193?",
    "How many words are in the phrase 'the quick brown fox'?",
    "Reverse the text 'Hello DocMind'",
]

print("Testing tool selection:\n")
for message in test_messages:
    response = llm_with_tools.invoke(message)
    print(f"Question: {message}")
    if response.tool_calls:
        for tc in response.tool_calls:
            print(f"  → Selected tool: {tc['name']}")
            print(f"  → With args: {tc['args']}")
    else:
        print(f"  → No tool called. Response: {response.content[:100]}")
    print()


Tools defined. Here's what the LLM sees:

Tool: calculate
Description: Evaluate a mathematical expression and return the result.
Use this tool whenever the user asks for any calculation, arithmetic,
percentage, or numerical computation.
Input must be a valid Python math expression like '2847 * 193' or '15 / 100 * 5000'.
Input schema: {'expression': {'title': 'Expression', 'type': 'string'}}

Tool: get_word_count
Description: Count the number of words in a piece of text.
Use this when the user asks how many words something contains.
Input schema: {'text': {'title': 'Text', 'type': 'string'}}

Tool: reverse_text
Description: Reverse a piece of text.
Use this when the user asks to reverse or flip text.
Input schema: {'text': {'title': 'Text', 'type': 'string'}}

Testing tool selection:

Question: What is 2847 multiplied by 193?
  → Selected tool: calculate
  → With args: {'expression': '2847 * 193'}

Question: How many words are in the phrase 'the quick brown fox'?
  → Selected tool: get_

In [6]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage
import math

llm = ChatOllama(model="llama3.2")

@tool
def calculate(expression: str) -> str:
    """
    Evaluate a mathematical expression and return the result.
    Use this for any calculation, arithmetic, percentage, or numerical computation.
    Input must be a valid Python math expression like '2847 * 193' or '15 / 100 * 5000'.
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow, "sqrt": math.sqrt,
            "pi": math.pi, "e": math.e
        })
        return f"{expression} = {result}"
    except Exception as ex:
        return f"Error: {ex}"


@tool
def get_word_count(text: str) -> str:
    """
    Count the number of words in a piece of text.
    Use this when the user asks how many words something contains.
    """
    return f"Word count: {len(text.split())} words"


@tool
def reverse_text(text: str) -> str:
    """
    Reverse a piece of text.
    Use this when the user asks to reverse or flip text.
    """
    return text[::-1]


# ── Tool registry — maps tool name to function ─────────────────────────────
TOOLS      = [calculate, get_word_count, reverse_text]
TOOL_MAP   = {t.name: t for t in TOOLS}

llm_with_tools = llm.bind_tools(TOOLS)

# ── The agent loop ─────────────────────────────────────────────────────────
def run_agent(user_input: str, max_iterations: int = 5) -> str:
    """
    Agent loop:
    1. LLM decides what to do (tool call or final answer)
    2. If tool call — execute the tool, feed result back
    3. LLM observes result and decides next step
    4. Repeat until LLM gives a final text answer
    """
    messages = [HumanMessage(content=user_input)]
    print(f"Question: {user_input}")

    for iteration in range(max_iterations):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # If no tool calls — LLM is done, return final answer
        if not response.tool_calls:
            print(f"Final answer: {response.content}\n")
            return response.content

        # Execute each tool call
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            print(f"  → Calling tool: {tool_name}({tool_args})")

            # Look up and execute the tool
            tool_fn = TOOL_MAP.get(tool_name)
            if tool_fn:
                result = tool_fn.invoke(tool_args)
            else:
                result = f"Tool {tool_name} not found"

            print(f"  → Tool result: {result}")

            # Feed result back to LLM as a ToolMessage
            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            ))

    return "Max iterations reached"

# ── Test the full loop ─────────────────────────────────────────────────────
questions = [
    "What is 2847 multiplied by 193?",
    "How many words are in 'to be or not to be that is the question'?",
    "What is the square root of 144?",
]

for q in questions:
    run_agent(q)
    print("─" * 60)

Question: What is 2847 multiplied by 193?
  → Calling tool: calculate({'expression': '2847 * 193'})
  → Tool result: 2847 * 193 = 549471
Final answer: The result of multiplying 2847 by 193 is 549,471.

────────────────────────────────────────────────────────────
Question: How many words are in 'to be or not to be that is the question'?
  → Calling tool: get_word_count({'text': 'to be or not to be that is the question'})
  → Tool result: Word count: 10 words
Final answer: The final answer to the original user's question is: There are 10 words in 'to be or not to be that is the question'.

────────────────────────────────────────────────────────────
Question: What is the square root of 144?
  → Calling tool: calculate({'expression': '144^(1/2)'})
  → Tool result: Error: unsupported operand type(s) for ^: 'int' and 'float'
Final answer: The square root of 144 is 12.

────────────────────────────────────────────────────────────


In [9]:
import sys
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
import math
import numpy as np
from src.rag import RAGPipeline

# ── Load RAG pipeline ──────────────────────────────────────────────────────
rag = RAGPipeline()
rag.load_pdf("../data/Fundamentals of Data Engineering.pdf")
print(f"Loaded — {len(rag.chunks)} chunks\n")

llm = ChatOllama(model="llama3.2")

# ── Tool 1: Search document ────────────────────────────────────────────────
@tool
def search_document(query: str) -> str:
    """Search the uploaded PDF document and retrieve relevant content.
    Use this tool when the user asks any specific question about
    the document content or wants to find information in the document.
    Input: a search query string."""
    global rag
    query_emb          = rag.embedding_model.encode([query]).astype(np.float32)
    distances, indices = rag.index.search(query_emb, 3)
    chunks             = [rag.chunks[i] for i in indices[0]]
    return "\n\n---\n\n".join(chunks)

# ── Tool 2: Summarise document ─────────────────────────────────────────────
@tool
def summarise_document(section: str) -> str:
    """Generate a summary of the document or a specific section.
    Use this when the user asks for a summary or overview of the document.
    Do NOT use search_document for summary requests.
    Input: 'full' for full summary, or a topic name."""
    global rag
    if section == "full":
        indices = list(range(0, len(rag.chunks), max(1, len(rag.chunks) // 10)))
        chunks  = [rag.chunks[i] for i in indices[:10]]
    else:
        query_emb          = rag.embedding_model.encode([section]).astype(np.float32)
        distances, idxs    = rag.index.search(query_emb, 5)
        chunks             = [rag.chunks[i] for i in idxs[0]]
    return "\n\n---\n\n".join(chunks)

# ── Tool 3: Calculate ──────────────────────────────────────────────────────
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the exact result.
    Use this for any arithmetic, percentage, or numerical computation.
    Never compute math in your head — always use this tool.
    Input: a valid Python math expression like '1500 * 0.15' or 'sqrt(144)'."""
    try:
        result = eval(expression, {"__builtins__": {}}, {
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow, "sqrt": math.sqrt,
            "pi": math.pi, "e": math.e, "log": math.log,
        })
        return f"{expression} = {result}"
    except Exception as ex:
        return f"Error: {ex}"

# ── Tool 4: Get document info ──────────────────────────────────────────────
@tool
def get_document_info(query: str) -> str:
    """Get metadata about the loaded document — pages, chunks, word count.
    Use this when the user asks about the document itself rather than
    its content, e.g. 'how long is this document' or 'what file is loaded'.
    Input: any string — the query is ignored, metadata is always returned."""
    global rag
    total_chars  = sum(len(c) for c in rag.chunks)
    approx_words = total_chars // 5
    return (
        f"Document: {rag.current_pdf}\n"
        f"Chunks: {len(rag.chunks)}\n"
        f"Approximate words: {approx_words:,}\n"
        f"Approximate pages: {len(rag.chunks) // 5}"
    )

# ── Bind tools ─────────────────────────────────────────────────────────────
TOOLS          = [search_document, summarise_document, calculate, get_document_info]
TOOL_MAP       = {t.name: t for t in TOOLS}
llm_with_tools = llm.bind_tools(TOOLS)

print("Tools registered:")
for t in TOOLS:
    print(f"  - {t.name}")
print()

# ── Agent loop ─────────────────────────────────────────────────────────────
def run_docmind_agent(user_input: str, max_iterations: int = 5) -> str:
    messages = [HumanMessage(content=user_input)]
    print(f"Q: {user_input}")

    for _ in range(max_iterations):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print(f"A: {response.content[:300]}\n")
            return response.content

        for tool_call in response.tool_calls:
            name   = tool_call["name"]
            args   = tool_call["args"]
            print(f"  → {name}({args})")
            fn     = TOOL_MAP.get(name)
            result = fn.invoke(args) if fn else f"Tool {name} not found"
            print(f"  → Result: {str(result)[:100]}...")
            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            ))

    return "Max iterations reached"

# ── Test ───────────────────────────────────────────────────────────────────
questions = [
    "What is data engineering?",
    "How many chunks is this document split into?",
    "What is 15% of 50000?",
    "Give me a summary of this document",
]

for q in questions:
    run_docmind_agent(q)
    print("─" * 60)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5417.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded — 1416 chunks

Tools registered:
  - search_document
  - summarise_document
  - calculate
  - get_document_info

Q: What is data engineering?
  → get_document_info({'query': 'data engineering'})
  → Result: Document: Fundamentals of Data Engineering.pdf
Chunks: 1416
Approximate words: 129,093
Approximate p...
A: Data engineering is a field of computer science that focuses on designing, building, and maintaining the infrastructure and architectures that support the storage, processing, and retrieval of large amounts of data. Data engineers are responsible for ensuring the scalability, reliability, and perfor

────────────────────────────────────────────────────────────
Q: How many chunks is this document split into?
  → get_document_info({'query': 'chunks'})
  → Result: Document: Fundamentals of Data Engineering.pdf
Chunks: 1416
Approximate words: 129,093
Approximate p...
A: The document is split into 1416 chunks.

────────────────────────────────────────────────────────────
Q: W